# financial fraud detection: from transactions to review decisions

by **andrés de león echavarri · A01234560**
repo: https://github.com/Andresdle18/Demo-ti2006b_A01234560

data in databricks: csv at `/Volumes/workspace/ti2006b_2_12/fraud_data/creditcard.csv`, delta table `workspace.ti2006b_2_12.creditcard`.

original notebook by dr. servio palacios.

the plan: load → audit → split by time → explore train only → fit models → pick model + threshold on validation → test once → log to mlflow.


## 1. setup

in databricks: workspace → import → file → pick this `.ipynb` → attach compute → run all.

needs python 3.11+ with pandas, numpy, matplotlib and scikit-learn.

upload the csv to a unity catalog volume and set `CSV_VOLUME_PATH` and `DELTA_TABLE_NAME` in the next cell. the notebook only creates the delta table if it doesn't exist yet.


In [ ]:
from pathlib import Path
import hashlib
import os
import json
import platform
import shutil
import sys
import tempfile
import time
import urllib.request

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, f1_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

SEED = 42
LOCAL_DATA_PATH = ''  # Local-only fallback; not for the Databricks submission.
CSV_VOLUME_PATH = '/Volumes/workspace/ti2006b_2_12/fraud_data/creditcard.csv'
DELTA_TABLE_NAME = 'workspace.ti2006b_2_12.creditcard'
OUTPUT_DIR = Path(tempfile.gettempdir()) / 'fraud_demo_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_MLFLOW = True  # Record the completed experiment in the Databricks workspace.
MLFLOW_EXPERIMENT = ''  # Databricks: supply an experiment path you can write to.

versions = {
    'python': platform.python_version(), 'numpy': np.__version__,
    'pandas': pd.__version__, 'scikit_learn': sklearn.__version__,
    'matplotlib': matplotlib.__version__,
}
display(pd.Series(versions, name='version').to_frame())
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.grid': False})


## 2. the dataset

284,807 card transactions from europe (sept 2013, two days), only 492 are fraud.

| column | what it is |
|---|---|
| `Time` | seconds since the first transaction, used to sort and split |
| `V1`–`V28` | anonymized pca features, no real business meaning |
| `Amount` | transaction amount, we use `log(1 + Amount)` |
| `Class` | 0 = legit, 1 = fraud. it's the label, never a feature |

the pca was already done by the source, so we just use the features as they come.


In [ ]:
DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'
EXPECTED_SHA256 = '76274b691b16a6c49d3f159c883398e03ccd6d1ee12d9d8ee38f4b4b98551a89'

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

ON_DATABRICKS = bool(os.environ.get('DATABRICKS_RUNTIME_VERSION'))
if ON_DATABRICKS:
    if not CSV_VOLUME_PATH or not DELTA_TABLE_NAME:
        raise ValueError('Set CSV_VOLUME_PATH and DELTA_TABLE_NAME to actual Unity Catalog paths.')
    data_path = Path(CSV_VOLUME_PATH).expanduser()
    if not data_path.is_file():
        raise FileNotFoundError(f'CSV volume path not found: {data_path}')
    volume_sdf = (spark.read.option('header', True).option('inferSchema', True).csv(CSV_VOLUME_PATH))
    csv_row_count = volume_sdf.count()
    if not spark.catalog.tableExists(DELTA_TABLE_NAME):
        volume_sdf.write.format('delta').mode('overwrite').saveAsTable(DELTA_TABLE_NAME)
    delta_sdf = spark.table(DELTA_TABLE_NAME)
    delta_row_count = delta_sdf.count()
    if csv_row_count != delta_row_count:
        raise ValueError(f'Row-count mismatch: volume CSV={csv_row_count:,}, Delta table={delta_row_count:,}.')
    print(f'CSV path: {CSV_VOLUME_PATH} ({csv_row_count:,} rows)')
    print(f'Delta table: {DELTA_TABLE_NAME} ({delta_row_count:,} rows)')
    raw = delta_sdf.toPandas()
else:
    if LOCAL_DATA_PATH:
        data_path = Path(LOCAL_DATA_PATH).expanduser()
        if not data_path.is_file():
            raise FileNotFoundError(f'CSV not found: {data_path}')
    elif Path('creditcard.csv').is_file():
        data_path = Path('creditcard.csv')
    else:
        data_path = Path(tempfile.gettempdir()) / 'fraud_demo_data' / 'creditcard.csv'
        data_path.parent.mkdir(parents=True, exist_ok=True)
        if not data_path.is_file():
            partial_path = data_path.with_suffix('.download')
            try:
                request = urllib.request.Request(DATA_URL, headers={'User-Agent': 'fraud-teaching-demo/1.0'})
                with urllib.request.urlopen(request, timeout=120) as response:
                    with partial_path.open('wb') as handle:
                        shutil.copyfileobj(response, handle)
                partial_path.replace(data_path)
            except Exception as exc:
                partial_path.unlink(missing_ok=True)
                raise RuntimeError('Download failed. Set LOCAL_DATA_PATH or an accessible Databricks volume path.') from exc
    raw = pd.read_csv(data_path)

dataset_sha256 = sha256_file(data_path)
if dataset_sha256 != EXPECTED_SHA256:
    raise ValueError('Dataset checksum differs from the verified snapshot. Check the file/source.')
print(f'Loaded {len(raw):,} rows × {raw.shape[1]} columns; SHA-256 verified.')

## 3. audit the data

we check that the columns are right, there are no missing or weird values, and we count exact duplicates. we keep the first copy of each duplicate.

we don't drop big amounts or strange values, those might be exactly the frauds we're looking for.


In [ ]:
expected_columns = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount', 'Class']
assert list(raw.columns) == expected_columns, 'Unexpected schema or column order.'
assert raw.shape == (284807, 31), 'Unexpected dataset dimensions.'
assert raw['Class'].value_counts().to_dict() == {0: 284315, 1: 492}
assert all(pd.api.types.is_numeric_dtype(raw[c]) for c in expected_columns)
assert not raw.isna().any().any(), 'Investigate missing data before modeling.'
assert np.isfinite(raw.to_numpy(dtype=float)).all(), 'Nonfinite numeric value found.'
assert raw['Class'].isin([0, 1]).all()
assert raw['Amount'].ge(0).all() and raw['Time'].ge(0).all()

duplicates = int(raw.duplicated().sum())
data = raw.drop_duplicates(keep='first').sort_values('Time', kind='stable').reset_index(drop=True)
audit = pd.DataFrame({
    'stage': ['original', 'after_exact_deduplication'],
    'rows': [len(raw), len(data)],
    'frauds': [int(raw['Class'].sum()), int(data['Class'].sum())],
    'missing_cells': [int(raw.isna().sum().sum()), int(data.isna().sum().sum())],
})
audit['fraud_rate'] = audit['frauds'] / audit['rows']
display(audit)
print(f'Exact duplicate rows removed: {duplicates:,}')


## 4. split by time

first 60% of the timeline goes to train, next 20% to validation, last 20% to test. same timestamp always stays in the same split.

- **train:** fit the models
- **validation:** compare models and pick the threshold
- **test:** check the final choice once

no smote or resampling, just class weights.


In [ ]:
train_cut, validation_cut = data['Time'].quantile([0.60, 0.80]).to_numpy()
train = data.loc[data['Time'] <= train_cut].copy()
validation = data.loc[(data['Time'] > train_cut) & (data['Time'] <= validation_cut)].copy()
test = data.loc[data['Time'] > validation_cut].copy()

split_summary = pd.DataFrame([
    {'split': name, 'rows': len(part), 'frauds': int(part['Class'].sum()),
     'fraud_rate': part['Class'].mean(), 'first_second': part['Time'].min(),
     'last_second': part['Time'].max()}
    for name, part in [('train', train), ('validation', validation), ('test', test)]
])
assert len(train) + len(validation) + len(test) == len(data)
assert train['Time'].max() < validation['Time'].min()
assert validation['Time'].max() < test['Time'].min()
assert all(part['Class'].nunique() == 2 for part in [train, validation, test])
display(split_summary)


## 5. look at the train data

quick look at amounts and class balance using train only.

think first: if only ~0.2% is fraud, a model that always says "legit" gets ~99.8% accuracy. does it catch any fraud? nope.


In [ ]:
display(train.groupby('Class')['Amount'].describe())
fig, axes = plt.subplots(1, 3, figsize=(14, 3.7))
counts = train['Class'].value_counts().reindex([0, 1])
axes[0].bar(['Legitimate', 'Fraud'], counts, color=['#2563eb', '#dc2626'])
axes[0].set_yscale('log')
axes[0].set_ylabel('Training transactions (log scale)')
axes[0].set_title('Severe class imbalance')
for label, color in [(0, '#2563eb'), (1, '#dc2626')]:
    values = np.log1p(train.loc[train['Class'] == label, 'Amount'])
    axes[1].hist(values, bins=35, density=True, alpha=0.5, color=color,
                 label='Fraud' if label else 'Legitimate')
axes[1].set(xlabel='log(1 + Amount)', ylabel='Density', title='Amount alone is not a fraud rule')
axes[1].legend()
scatter = train.sample(n=min(10000, len(train)), random_state=SEED)
for label, color in [(0, '#2563eb'), (1, '#dc2626')]:
    points = scatter.loc[scatter['Class'] == label]
    axes[2].scatter(points['V14'], points['V17'], s=12, alpha=0.45,
                    color=color, label='Fraud' if label else 'Legitimate')
axes[2].set(xlabel='V14 (PCA)', ylabel='V17 (PCA)', title='10,000 sampled training rows')
axes[2].legend()
fig.tight_layout()
plt.show()
print(f'Always-legitimate training accuracy: {1 - train["Class"].mean():.4%}; fraud recall: 0%.')


## 6. features

we use `V1`–`V28` plus `LogAmount`. `Class` is the label and `Time` only sets the order, so neither one goes in as a feature.


In [ ]:
FEATURES = [f'V{i}' for i in range(1, 29)] + ['LogAmount']

def make_features(frame):
    features = frame[[f'V{i}' for i in range(1, 29)]].copy()
    features['LogAmount'] = np.log1p(frame['Amount'])
    assert list(features.columns) == FEATURES
    assert np.isfinite(features.to_numpy()).all()
    return features

X_train, y_train = make_features(train), train['Class'].astype(int)
X_validation, y_validation = make_features(validation), validation['Class'].astype(int)
assert 'Class' not in FEATURES and 'Time' not in FEATURES
print('Predictor matrix:', X_train.shape)


## 7. models

| model | why |
|---|---|
| always legit | baseline that shows why accuracy lies |
| logistic regression | simple linear baseline, scaled, balanced weights |
| decision tree | easy to explain, depth 4, min 20 rows per leaf |
| random forest | 100 trees, depth 12, min 5 rows per leaf |
| isolation forest | unsupervised, flags weird points |

trees split on whatever lowers gini (1 − p₀² − p₁²) the most. a pure node has gini 0. balanced class weights make the few frauds count more.


In [ ]:
models = {
    'Always legitimate': DummyClassifier(strategy='constant', constant=0),
    'Logistic regression': Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1500,
                                     solver='lbfgs', random_state=SEED)),
    ]),
    'Decision tree': DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=20, class_weight='balanced', random_state=SEED),
    'Random forest': RandomForestClassifier(
        n_estimators=100, max_depth=12, min_samples_leaf=5,
        class_weight='balanced_subsample', n_jobs=2, random_state=SEED),
    'Isolation Forest': IsolationForest(
        n_estimators=100, max_samples=256, contamination='auto',
        n_jobs=2, random_state=SEED),
}

fit_seconds = {}
for name, model in models.items():
    started = time.perf_counter()
    if name == 'Isolation Forest':
        model.fit(X_train)  # No labels are supplied.
    else:
        model.fit(X_train, y_train)
    fit_seconds[name] = time.perf_counter() - started
display(pd.Series(fit_seconds, name='fit_seconds').to_frame().round(2))


## 8. metrics that actually matter

- **precision:** of the alerts, how many are real fraud
- **recall:** of the frauds, how many we caught
- **f1:** balance between the two
- **ap:** precision-recall summary across all thresholds
- **roc-auc:** how well it ranks fraud above legit
- **alert rate:** % of transactions sent to review

first we compare on validation at the default threshold (0.5).


In [ ]:
def risk_scores(name, model, features):
    if name == 'Isolation Forest':
        return -model.score_samples(features)
    fraud_column = int(np.flatnonzero(model.classes_ == 1)[0])
    return model.predict_proba(features)[:, fraud_column]

def operating_metrics(labels, scores, threshold):
    predictions = (np.asarray(scores) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {
        'threshold': float(threshold), 'AP': average_precision_score(labels, scores),
        'ROC_AUC': roc_auc_score(labels, scores),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'F1': f1_score(labels, predictions, zero_division=0),
        'accuracy': accuracy_score(labels, predictions), 'alert_rate': predictions.mean(),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
    }

validation_scores = {}
default_thresholds = {}
validation_rows = []
for name, model in models.items():
    scores = risk_scores(name, model, X_validation)
    validation_scores[name] = scores
    threshold = -float(model.offset_) if name == 'Isolation Forest' else 0.5
    default_thresholds[name] = threshold
    validation_rows.append({'model': name, **operating_metrics(y_validation, scores, threshold)})
validation_metrics = pd.DataFrame(validation_rows).set_index('model')
display(validation_metrics.sort_values('AP', ascending=False).round(5))

fig, ax = plt.subplots(figsize=(8, 4.5))
for name, scores in validation_scores.items():
    precision, recall, _ = precision_recall_curve(y_validation, scores)
    ax.step(recall, precision, where='post',
            label=f'{name}: AP={average_precision_score(y_validation, scores):.3f}')
ax.axhline(y_validation.mean(), color='gray', linestyle='--', label='Validation prevalence')
ax.set(xlabel='Recall', ylabel='Precision', title='Validation precision–recall curves',
       xlim=(0, 1), ylim=(0, 1.02))
ax.legend(loc='lower left', fontsize=8)
fig.tight_layout()
plt.show()


## 9. read the tree

left branch = the condition is true. the plot shows the first levels and the text shows the full rules. feature importance just says which splits helped most, it doesn't prove causality.


In [ ]:
tree_model = models['Decision tree']
fig, ax = plt.subplots(figsize=(17, 7))
plot_tree(tree_model, feature_names=FEATURES, class_names=['Legitimate', 'Fraud'],
          filled=True, rounded=True, max_depth=2, fontsize=8, ax=ax)
ax.set_title('Fitted depth-4 decision tree: first three levels shown')
fig.tight_layout()
plt.show()
print(export_text(tree_model, feature_names=FEATURES, decimals=3))
tree_importance = pd.Series(tree_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
display(tree_importance.head(10).rename('impurity_importance').to_frame())


## 10. pick model and threshold (validation only)

rule: the supervised model with the highest validation ap wins.

threshold: at least 80% recall on validation, and among those, the one with the best precision.

validation has few frauds, so the threshold has some uncertainty.


In [ ]:
supervised_candidates = ['Logistic regression', 'Decision tree', 'Random forest']
selected_name = validation_metrics.loc[supervised_candidates, 'AP'].idxmax()
selected_model = models[selected_name]
TARGET_RECALL = 0.80

precision, recall, thresholds = precision_recall_curve(
    y_validation, validation_scores[selected_name])
threshold_table = pd.DataFrame({
    'threshold': thresholds, 'precision': precision[:-1], 'recall': recall[:-1],
})
feasible = threshold_table.loc[threshold_table['recall'] >= TARGET_RECALL]
if feasible.empty:
    raise RuntimeError('No validation threshold satisfies the declared recall policy.')
chosen = feasible.sort_values(['precision', 'threshold'], ascending=[False, False]).iloc[0]
selected_threshold = float(chosen['threshold'])
validation_selected = operating_metrics(
    y_validation, validation_scores[selected_name], selected_threshold)
assert validation_selected['recall'] >= TARGET_RECALL
print('Selected model:', selected_name)
display(pd.Series(validation_selected, name='validation_at_frozen_threshold').to_frame())

fig, ax = plt.subplots(figsize=(8, 3.7))
ax.plot(threshold_table['threshold'], threshold_table['precision'], label='Validation precision')
ax.plot(threshold_table['threshold'], threshold_table['recall'], label='Validation recall')
ax.axvline(selected_threshold, color='black', linestyle='--', label='Frozen threshold')
ax.axhline(TARGET_RECALL, color='gray', linestyle=':')
ax.set(xlabel='Risk threshold', ylabel='Metric', title=f'{selected_name}: policy trade-off', ylim=(0, 1.03))
ax.legend()
fig.tight_layout()
plt.show()


## 11. test once

first time we touch test. we compare the dummy, the chosen model at 0.5 and the chosen model at our threshold. we don't change anything after seeing this.

- false negatives = frauds we missed
- false positives = legit transactions sent to review

intervals are wilson 95%.


In [ ]:
X_test, y_test = make_features(test), test['Class'].astype(int)
test_scores = risk_scores(selected_name, selected_model, X_test)
test_results = operating_metrics(y_test, test_scores, selected_threshold)
test_default = operating_metrics(y_test, test_scores, default_thresholds[selected_name])
dummy_scores = risk_scores('Always legitimate', models['Always legitimate'], X_test)
test_baseline = operating_metrics(y_test, dummy_scores, 0.5)
test_comparison = pd.DataFrame([
    {'policy': 'Always legitimate', **test_baseline},
    {'policy': f'{selected_name}: default', **test_default},
    {'policy': f'{selected_name}: frozen validation policy', **test_results},
]).set_index('policy')
display(test_comparison.round(5))

test_predictions = (test_scores >= selected_threshold).astype(int)
fig, ax = plt.subplots(figsize=(5.5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, labels=[0, 1], display_labels=['Legitimate', 'Fraud'],
    cmap='Blues', colorbar=False, values_format='d', ax=ax)
ax.set_title('Future test: frozen validation policy')
fig.tight_layout()
plt.show()

def wilson_interval(successes, trials, z=1.96):
    if trials == 0:
        return (float('nan'), float('nan'))
    p = successes / trials
    denominator = 1 + z * z / trials
    center = (p + z * z / (2 * trials)) / denominator
    margin = z * np.sqrt(p * (1 - p) / trials + z * z / (4 * trials * trials)) / denominator
    return float(center - margin), float(center + margin)

recall_interval = wilson_interval(test_results['TP'], test_results['TP'] + test_results['FN'])
precision_interval = wilson_interval(test_results['TP'], test_results['TP'] + test_results['FP'])
display(pd.DataFrame([
    {'metric': 'recall', 'estimate': test_results['recall'], 'lower_95': recall_interval[0], 'upper_95': recall_interval[1]},
    {'metric': 'precision', 'estimate': test_results['precision'], 'lower_95': precision_interval[0], 'upper_95': precision_interval[1]},
]))


## 12. workload and cost

alerts per 10k transactions. made-up cost: missed fraud = 100, false alert = 1. it's just to talk about tradeoffs, not a real business case.


In [ ]:
COST_MISSED_FRAUD = 100.0
COST_FALSE_ALERT = 1.0
cost_scenario = test_comparison[['FP', 'FN', 'TP', 'alert_rate']].copy()
cost_scenario['alerts_per_10000'] = 10000 * cost_scenario['alert_rate']
cost_scenario['illustrative_cost'] = (
    COST_MISSED_FRAUD * cost_scenario['FN'] + COST_FALSE_ALERT * cost_scenario['FP'])
display(cost_scenario.round(2))
print(f'Frozen policy sends {test_predictions.sum():,} of {len(test):,} transactions for review.')


## 13. score new transactions

small function that applies the same features and threshold and returns a risk score plus a review flag. we demo it on 5 test rows.


In [ ]:
def score_transactions(frame):
    required = [f'V{i}' for i in range(1, 29)] + ['Amount']
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    if not all(pd.api.types.is_numeric_dtype(frame[c]) for c in required):
        raise ValueError('Required inputs must have numeric dtypes.')
    values = frame[required].to_numpy(dtype=float)
    if not np.isfinite(values).all() or frame['Amount'].lt(0).any():
        raise ValueError('Inputs must be finite and Amount must be nonnegative.')
    scores = risk_scores(selected_name, selected_model, make_features(frame))
    return pd.DataFrame({
        'risk_score': scores, 'review_flag': (scores >= selected_threshold).astype(int),
    }, index=frame.index)

demo_input = test.head(5).drop(columns=['Class', 'Time'])
demo_scores = score_transactions(demo_input)
assert np.allclose(demo_scores['risk_score'], test_scores[:5])
display(demo_scores)


## 14. export results

saves the audit, splits, metrics, test scores and a manifest (dataset hash, versions, threshold) as csv/json.


In [ ]:
audit.to_csv(OUTPUT_DIR / 'data_audit.csv', index=False)
split_summary.to_csv(OUTPUT_DIR / 'split_summary.csv', index=False)
validation_metrics.to_csv(OUTPUT_DIR / 'validation_metrics.csv')
test_comparison.to_csv(OUTPUT_DIR / 'test_metrics.csv')
scored_test = pd.DataFrame({
    'source_row_after_dedup': test.index, 'Time': test['Time'].to_numpy(),
    'Amount': test['Amount'].to_numpy(), 'actual_class': y_test.to_numpy(),
    'risk_score': test_scores, 'review_flag': test_predictions,
})
scored_test.to_csv(OUTPUT_DIR / 'test_scores.csv', index=False)
manifest = {
    'dataset_url': DATA_URL, 'dataset_sha256': dataset_sha256,
    'original_rows': len(raw), 'deduplicated_rows': len(data),
    'duplicate_policy': 'keep first exact row', 'seed': SEED,
    'split': 'chronological quantiles 60/20/20; tied timestamps kept together',
    'train_cut_seconds': float(train_cut), 'validation_cut_seconds': float(validation_cut),
    'features': FEATURES, 'versions': versions, 'selected_model': selected_name,
    'selected_model_parameters': {
        key: value for key, value in selected_model.get_params().items()
        if value is None or isinstance(value, (str, bool, int, float))
    },
    'selection_metric': 'validation average precision, supervised candidates only',
    'target_validation_recall': TARGET_RECALL, 'selected_threshold': selected_threshold,
    'validation_metrics_at_selected_threshold': validation_selected,
    'test_metrics_at_selected_threshold': test_results,
    'test_recall_wilson_95': recall_interval,
    'test_precision_wilson_95': precision_interval,
    'cost_assumptions': {'missed_fraud': COST_MISSED_FRAUD, 'false_alert': COST_FALSE_ALERT},
}
with (OUTPUT_DIR / 'run_manifest.json').open('w') as handle:
    json.dump(manifest, handle, indent=2, allow_nan=False)
print('Artifacts:', OUTPUT_DIR)
print('\n'.join(sorted(path.name for path in OUTPUT_DIR.iterdir() if path.is_file())))


## 15. mlflow

logs the run: params (model, seed, threshold, dataset hash, split), test metrics, the fitted model and the result files. it's experiment tracking, not deployment.


In [ ]:
if RUN_MLFLOW:
    import os
    import mlflow
    import mlflow.sklearn
    from mlflow.models import infer_signature

    if not os.environ.get('DATABRICKS_RUNTIME_VERSION') and not os.environ.get('MLFLOW_TRACKING_URI'):
        mlflow.set_tracking_uri((Path(tempfile.gettempdir()) / 'fraud_demo_mlflow').as_uri())
    if MLFLOW_EXPERIMENT:
        mlflow.set_experiment(MLFLOW_EXPERIMENT)
    elif not os.environ.get('DATABRICKS_RUNTIME_VERSION'):
        mlflow.set_experiment('financial_fraud_teaching_demo')

    with mlflow.start_run(run_name='chronological_fraud_demo'):
        mlflow.log_params({
            'selected_model': selected_name, 'seed': SEED,
            'review_threshold': selected_threshold, 'validation_recall_target': TARGET_RECALL,
            'dataset_sha256': dataset_sha256, 'split_policy': 'chronological_60_20_20',
        })
        mlflow.log_metrics({f'test_{key}': float(value) for key, value in test_results.items()})
        signature = infer_signature(X_train.head(5), selected_model.predict(X_train.head(5)))
        mlflow.sklearn.log_model(
            selected_model, artifact_path='model', signature=signature,
            input_example=X_train.head(5),
        )
        mlflow.log_artifact(str(OUTPUT_DIR / 'run_manifest.json'))
        mlflow.log_artifact(str(OUTPUT_DIR / 'validation_metrics.csv'))
        mlflow.log_artifact(str(OUTPUT_DIR / 'test_metrics.csv'))
        print('MLflow run:', mlflow.active_run().info.run_id)
else:
    print('Optional MLflow logging is disabled; core demo is complete.')


## 16. questions

answer with your own run's results.

1. dummy accuracy, recall and ap: why do they tell different stories?
2. walk one path in the tree. what do gini, depth, leaf size and class weights do?
3. which model won validation ap? is the best roc-auc always the best pick?
4. default vs chosen threshold on test: how did fp and fn change? why can 80% recall miss on test?
5. why is isolation forest unsupervised but its evaluation isn't? is every weird transaction fraud?
6. why are filling missing values with 0, balancing test, or scaling before the split bad ideas?
7. duplicates, pca limits, no customer ids, privacy: what do you need before using this for real?
8. precision/recall intervals and alert load: can a team handle it? why isn't the cost scenario a business case?


## my answers — andrés de león echavarri (A01234560)

_pending: i fill this in after running it in databricks._


## 17. References and dataset attribution

1. Worldline and Machine Learning Group, Université Libre de Bruxelles.
   *Credit Card Fraud Detection* (original dataset listing):
   https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
   Review the original dataset's current usage/license terms before redistribution.
2. TensorFlow. *Classification on imbalanced data*. Official description and public CSV mirror:
   https://www.tensorflow.org/tutorials/structured_data/imbalanced_data
   CSV: https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv
3. Lucas, Y., and Jurgovsky, J. (2020). *Credit card fraud detection using machine learning:
   A survey*. arXiv:2010.06479. https://arxiv.org/abs/2010.06479
   Background for imbalance, evaluation, temporal behavior, and concept drift.
4. Niu, X., Wang, L., and Yang, X. (2019). *A Comparison Study of Credit Card Fraud Detection:
   Supervised versus Unsupervised*. arXiv:1904.10604. https://arxiv.org/abs/1904.10604
   Motivation for supervised/unsupervised comparisons; this notebook does not reproduce its protocol/results.
5. scikit-learn. *Decision Trees*. https://scikit-learn.org/stable/modules/tree.html
6. scikit-learn. *IsolationForest* API and original algorithm references.
   https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
7. scikit-learn. *Precision-Recall*.
   https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html
8. scikit-learn. *Tuning the decision threshold for class prediction*.
   https://scikit-learn.org/stable/modules/classification_threshold.html
9. Databricks. *Import and export Databricks notebooks*.
   https://docs.databricks.com/aws/en/notebooks/notebook-export-import
10. Original Databricks PaySim example supplied for context:
    https://pages.databricks.com/rs/094-YMS-629/images/financial-fraud-detection-decision-tree.html

Sources and dataset access checked November 12, 2026. All modeling code in this notebook is
a new teaching implementation; published benchmark scores are not presented as our results.
